# Python Decorator Examples

This notebook builds up from plain Python decorators to Airflow's `@dag` / `@task` TaskFlow API used in `hello_world_gcs.py`.

## 1. A decorator is just a function that wraps a function

`@decorator` above a function definition is shorthand for `func = decorator(func)`.

In [1]:
def shout(func):
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        return result.upper()
    return wrapper


@shout
def greet(name):
    return f"hello, {name}"


print(greet("wilson"))

HELLO, WILSON


In [2]:
# Same thing without the @ syntax, to show it's just function composition
def greet_plain(name):
    return f"hello, {name}"


greet_plain = shout(greet_plain)
print(greet_plain("wilson"))

HELLO, WILSON


## 2. Decorators that take arguments

Airflow's `@dag(dag_id=..., schedule=..., ...)` is a decorator *factory* — a function that returns a decorator. This is different from `@task`, which is used bare.

Pattern: `@decorator(args)` needs an extra layer of nesting.

In [ ]:
def repeat(times):
    def decorator(func):
        def wrapper(*args, **kwargs):
            for _ in range(times):
                func(*args, **kwargs)
        return wrapper
    return decorator


@repeat(times=3)
def say_hi():
    print("hi")


say_hi()

Equivalent without `@` syntax — note the two calls: `repeat(times=3)` returns `decorator`, then `decorator(say_hi_plain)` returns `wrapper`.

In [ ]:
def say_hi_plain():
    print("hi")


say_hi_plain = repeat(times=3)(say_hi_plain)
say_hi_plain()

## 3. A decorator used bare (no parentheses)

This is the `@task` style — no arguments, so no extra nesting layer is needed.

In [ ]:
def log_call(func):
    def wrapper(*args, **kwargs):
        print(f"calling {func.__name__}")
        return func(*args, **kwargs)
    return wrapper


@log_call
def add(a, b):
    return a + b


print(add(2, 3))

## 4. A decorator that delays/registers instead of running immediately

This is the key trick behind Airflow's `@task`: calling a decorated function doesn't run it right away — it records something (a task node) and returns a placeholder. This mimics how `write_to_gcs()` inside a `@dag` function builds a graph node instead of executing immediately.

In [ ]:
registered_tasks = []


class TaskNode:
    def __init__(self, func):
        self.func = func
        self.name = func.__name__
        self.downstream = []

    def __rshift__(self, other):
        # implements the `>>` operator, like Airflow's task dependency syntax
        self.downstream.append(other)
        return other

    def __repr__(self):
        return f"TaskNode({self.name})"


def fake_task(func):
    def builder(*args, **kwargs):
        node = TaskNode(func)
        registered_tasks.append(node)
        return node
    return builder


@fake_task
def write_to_gcs():
    print("writing...")


@fake_task
def read_from_gcs():
    print("reading...")


write_task = write_to_gcs()
read_task = read_from_gcs()
write_task >> read_task

print("registered tasks:", registered_tasks)
print("write_task downstream:", write_task.downstream)

Notice: calling `write_to_gcs()` did **not** print "writing..." — it just created and registered a `TaskNode`. This is exactly the behavior of Airflow's `@task` decorator, and it's why in `hello_world_gcs.py` you can write `write_to_gcs() >> read_from_gcs()` to declare ordering instead of actually running the functions in sequence.

## 5. Putting it together: a mini `@dag` + `@task`

A simplified version of what Airflow does, so the real `hello_world_gcs.py` code reads naturally afterward.

In [ ]:
def mini_task(func):
    def builder(*args, **kwargs):
        node = TaskNode(func)
        return node
    return builder


def mini_dag(dag_id):
    def decorator(func):
        def builder():
            print(f"building DAG '{dag_id}'")
            func()  # runs the body, which wires up @mini_task calls and >>
            return dag_id
        return builder
    return decorator


@mini_dag(dag_id="hello_world_gcs")
def hello_world_gcs():
    @mini_task
    def write_to_gcs():
        print("writing...")

    @mini_task
    def read_from_gcs():
        print("reading...")

    write_to_gcs() >> read_from_gcs()


hello_world_gcs()  # calling it triggers the DAG-building process, like line 48 of hello_world_gcs.py

## 6. Back to the real file

```python
@dag(
    dag_id="hello_world_gcs",
    schedule=None,
    start_date=pendulum.datetime(2024, 1, 1, tz="UTC"),
    catchup=False,
    tags=["gcs", "example"],
)
def hello_world_gcs():

    @task
    def write_to_gcs() -> None:
        ...

    @task
    def read_from_gcs() -> None:
        ...

    write_to_gcs() >> read_from_gcs()


hello_world_gcs()
```

- `@dag(...)` is a decorator factory (like `mini_dag` / `repeat` above) — it takes config and returns a decorator that turns `hello_world_gcs` into a DAG builder.
- `@task` is a bare decorator (like `fake_task` / `log_call` above) — it turns each inner function into a task node instead of a function that runs immediately.
- `write_to_gcs() >> read_from_gcs()` uses the overloaded `>>` operator (like `TaskNode.__rshift__` above) to declare "run write before read", not to actually execute them in order right there.
- The final `hello_world_gcs()` call at the bottom of the file is what actually triggers DAG construction — Airflow's DAG file parser imports the module and that call registers the DAG.